<a href="https://colab.research.google.com/github/Bitikameza/EK2026/blob/main/Session_IV_Deep_Survival_Modeling_Using_PyCox_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Session IV: Deep Survival Models: [**`pycox`**](https://github.com/havakv/pycox)**

`PyCox` trains survival models under the hood, automating the training loop.

- **Discrete-Time Models**:

    - **`LogisticHazard`** – Discretizes time and models hazard via logistic regression (BCE with hazard parameterization).
    
    - **`PCHazard`** – Piecewise Constant Hazard model (discretized time and constant hazard intervals) (Possion/Loglikelihood loss).
    
    - **`PMF`** – Probability mass function model, predicting event time distribution directly (base for `DeepHit` and `MTLR`) (Categorical Cross-Entropy).

    - **`DeepHit`** – A deep learning model for competing risks (multiple event types) survival data (`DeepHitSingle` – simplified for a single risk).
    
    - **`BCESurv`** – Binary cross-entropy–based survival model, similar to `PMF` but trained with BCE loss (Categorical Cross-Entropy).

    - **`MTLR` (N-MTLR)** – (Neural) Multi-task logistic regression for survival analysis (Cross-Entropy Loss over time intervals).

- **Continuous-Time Models**:

    - **`CoxTime`** – A generalization of CoxPH where the network can model time-varying effects (Negative partial log-likelihood - Cox).

    - **`CoxCC`** – Cox model with a case–control sampling scheme (proportional version of Cox-Time model) (Negative partial log-likelihood - Cox).

    - **`CoxPH` (DeepSurv)** – Deep neural network extension of the Cox Proportional Hazards model (Negative log partial likelihood).


# **I. Discrete-Time Models**

## **1. Import Libraries and Load Dataset**

In [ ]:
!pip install pycox -q

In [ ]:
import torch
import pycox
import torchtuples as tt

In [ ]:
# import pycox.models as models
# print(dir(models))

In [ ]:
from pycox.models import CoxPH, LogisticHazard, PCHazard, PMF, MTLR, BCESurv, DeepHitSingle, CoxCC, CoxTime

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
url ="https://raw.githubusercontent.com/awolseid/Datasets/refs/heads/main/Survival_HAART.csv"
haart_df = pd.read_csv(url)
haart_df

## **2. Separate Input and Output Features Data**

In [ ]:
X_df = haart_df.drop(columns=["CardNum", "Defaulter", "SurvTime"])
Y_df = haart_df[["Defaulter", "SurvTime"]]
# Y_df

## **3. Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train_df, X_test_df, Y_train_df, Y_test_df = train_test_split(X_df, Y_df, stratify=Y_df["Defaulter"], test_size=0.15, random_state=42)
X_train_df, X_val_df, Y_train_df, Y_val_df = train_test_split(X_train_df, Y_train_df, stratify=Y_train_df["Defaulter"], test_size=0.15, random_state=42)

In [ ]:
X_train_df.shape, X_val_df.shape, X_test_df.shape

In [ ]:
X_val_df

## **4. Data Preprocessing**

### **4.1. Input Features Transformation**

**Identify Quantitative and Categorical Covariates**

In [ ]:
quant_features = list(X_df.select_dtypes(exclude="object").columns)
quant_features

In [ ]:
cat_features = list(X_df.select_dtypes(include="object").columns)
cat_features

**Tranform Features**

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

In [ ]:
cat_transformer = OneHotEncoder(drop="first")
quant_transformer = StandardScaler()

transformer = ColumnTransformer([
    ("CategoricalEncoder", cat_transformer, cat_features),
    ("QuantitativeScaler", quant_transformer, quant_features)])
transformer

- **Note** that PyTorch require variables of type `float32`.

In [ ]:
X_train = transformer.fit_transform(X_train_df).astype('float32')
X_val = transformer.transform(X_val_df).astype('float32')
X_test = transformer.transform(X_test_df).astype('float32')

### **4.2. Processing Outcomes**

#### **i. Tuplifying Event and Survival Time for Simplicity**

- Let us convert the outcomes data frame in to a tuple

In [ ]:
def tuplify_outcomes(Y_df, event_name, time_name):
    return np.array(Y_df[time_name]), np.array(Y_df[event_name])

In [ ]:
Y_train = tuplify_outcomes(Y_df=Y_train_df, event_name="Defaulter", time_name="SurvTime")
Y_train

In [ ]:
Y_val = tuplify_outcomes(Y_df=Y_val_df, event_name="Defaulter", time_name="SurvTime")
# Y_val

In [ ]:
Y_test = tuplify_outcomes(Y_df=Y_test_df, event_name="Defaulter", time_name="SurvTime")
# Y_test

#### **ii. Label Transformation: `.label_transform()`**

- Discrete-time survival models require discretization of the event times in to a number of intervals.

    - The size of the equidistant discretization grid defines the number of durations.
    
    - The network will have that number of durations output nodes.

- The function `.label_transform()` is used transforming individual labels

##### **I. Equidistant Discretization**

In [ ]:
discrete_times = 10

**Label Transform Instantiation**

In [ ]:
lh_labtrans = LogisticHazard.label_transform(discrete_times)
pch_labtrans = PCHazard.label_transform(discrete_times)
pmf_labtrans = PMF.label_transform(discrete_times)
mtlr_labtrans = MTLR.label_transform(discrete_times)
bces_labtrans = BCESurv.label_transform(discrete_times)
dhs_labtrans = DeepHitSingle.label_transform(discrete_times)

**i. Do Label Transformation: Train Set**

In [ ]:
Y_train_lh = lh_labtrans.fit_transform(*Y_train)
Y_train_pch = pch_labtrans.fit_transform(*Y_train)
Y_train_pmf = pmf_labtrans.fit_transform(*Y_train)
Y_train_mtlr = mtlr_labtrans.fit_transform(*Y_train)
Y_train_bces = bces_labtrans.fit_transform(*Y_train)
Y_train_dhs = dhs_labtrans.fit_transform(*Y_train)

- Number of output features

In [ ]:
lh_labtrans.out_features

- The discretization grid will be used to obtain the time-scale for survival predictions.

In [ ]:
lh_labtrans.cuts

In [ ]:
pch_labtrans.cuts

In [ ]:
pmf_labtrans.cuts

In [ ]:
mtlr_labtrans.cuts

In [ ]:
bces_labtrans.cuts

In [ ]:
dhs_labtrans.cuts

In [ ]:
Y_train_lh

- Visualization of the Kaplan-Meier estimates over the grid.

In [ ]:
from pycox.utils import kaplan_meier as KM
import matplotlib.pyplot as plt

In [ ]:
plt.vlines(lh_labtrans.cuts, 0, 1, colors='gray', linestyles="--", label='Discretization Grid')
KM(*Y_train).plot(label='Kaplan-Meier')
plt.ylabel('Survival Probability')
plt.xlabel('Time')
plt.legend();

- For each model above, the transformed label `Y_train` is a tuple with indices of the discretized times and event indicators.

In [ ]:
Y_train_lh

**ii. Do Label Transformation: Validation Set**

In [ ]:
Y_val_lh = lh_labtrans.fit_transform(*Y_val)
Y_val_pch = pch_labtrans.fit_transform(*Y_val)
Y_val_pmf = pmf_labtrans.fit_transform(*Y_val)
Y_val_mtlr = mtlr_labtrans.fit_transform(*Y_val)
Y_val_bces = bces_labtrans.fit_transform(*Y_val)
Y_val_dhs = dhs_labtrans.fit_transform(*Y_val)

**iii.** There is **no need to transform the test labels**

##### **II. Quantiles-Based Discretization**

- In quantiles-based discretization, the grid would be finer where more events are available and coarser where there are some.

In [ ]:
labtrans = LogisticHazard.label_transform(discrete_times, scheme='quantiles') # default is 'uniform'
labtrans.fit_transform(*Y_train)
labtrans.cuts

- The quantile discretization grid is different from equidistant discretization.

In [ ]:
plt.vlines(labtrans.cuts, 0, 1, colors='gray', linestyles="--", label='Discretization Grid')
KM(*Y_train).plot(label='Kaplan-Meier')
plt.ylabel('Survival Probability')
plt.xlabel('Time')
plt.legend();

## **5. Model Building**

### **i. Define the Network Architecture and Instantiate It**

- It is common construct custom DNN models subclassing `nn.Module` as we have seen in the previous notebook.

- Here, just for simplicity, we are going to use the `MLPVanilla()` architecture provided by `torchtuples`.

  `tt.practical.MLPVanilla(in_features, num_nodes, out_features, batch_norm, dropout)`

- Two hidden layers (with 32 nodes each), ReLU activations, and out_features output nodes.

- We also have batch **normalization** and **dropout** between the layers.

- Number of input features

In [ ]:
num_inputs = X_train.shape[1]
num_inputs

- Number of output features

In [ ]:
num_outputs = lh_labtrans.out_features
num_outputs

- Let the number of hidden layers be 2, each with 64 and 32 nodes, respectively.

In [ ]:
hidden_nodes = [64, 32]
hidden_nodes

- There are also other parameters like activation, batch normalization, dropout, etc.

- We are not going to specify them, rather use their default values.

In [ ]:
dnn_model = tt.practical.MLPVanilla(
    in_features  = num_inputs,
    num_nodes    = hidden_nodes,
    out_features = num_outputs)
dnn_model

**Example:** Define the above architecture in a custom model subclassing the `nn.module` of PyTorch.

### **ii. Specify the Optimizer**

- Use from `import torch.nn.optim as optim`

- Specify the `SGD`, or `Adam` optimizer with learning rate `0.01`.

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
optimizer

### **iii. Instantiate the Survival Model**

- The `duration_index` option connects the output nodes of the network the the discretization times.

- This is **only useful for prediction** and does not affect the training procedure.

In [ ]:
lh_model   = LogisticHazard(dnn_model, optimizer, duration_index=lh_labtrans.cuts)
pch_model  = PCHazard(dnn_model,       optimizer, duration_index=pch_labtrans.cuts)
pmf_model  = PMF(dnn_model,            optimizer, duration_index=pmf_labtrans.cuts)
mtlr_model = MTLR(dnn_model,           optimizer, duration_index=mtlr_labtrans.cuts)
bces_model = BCESurv(dnn_model,        optimizer, duration_index=bces_labtrans.cuts)
dhs_model  = DeepHitSingle(dnn_model,  optimizer, duration_index=dhs_labtrans.cuts)

### **iii. Train Models**

- Specify the `batch_size` and the number of training `epochs`.

In [ ]:
batch_size = 128
epochs     = 100

- Train model
  
    - Include `EarlyStopping` callback to stop training when validation loss stops improving.
      
    - Note both the DNN and survival models need to be instantiated in every training.

In [ ]:
torch.manual_seed(42)

dnn_model = tt.practical.MLPVanilla(in_features  = num_inputs,
                                    num_nodes    = hidden_nodes,
                                    out_features = num_outputs)
optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
lh_model   = LogisticHazard(dnn_model, optimizer, duration_index=lh_labtrans.cuts)
callbacks  = [tt.cb.EarlyStopping()]
lh_log = lh_model.fit(X_train, Y_train_lh, batch_size, epochs, callbacks, val_data=(X_val, Y_val_lh))

- The `log` object keeps track of the training progress.

In [ ]:
lh_log.to_pandas()

In [ ]:
lh_log.plot();

- After termination, the `EarlyStopping` callback loads the best performing model (in terms of validation loss).

- We can verify this by comparing the minimum validation loss to the validation score of the trained model.

In [ ]:
lh_log.to_pandas().val_loss.min()

In [ ]:
lh_model.score_in_batches(X_val, Y_val_lh)

In [ ]:
torch.manual_seed(42)

dnn_model = tt.practical.MLPVanilla(in_features  = num_inputs,
                                    num_nodes    = hidden_nodes,
                                    out_features = num_outputs)
optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
pch_model  = PCHazard(dnn_model, optimizer, duration_index=pch_labtrans.cuts)
callbacks  = [tt.cb.EarlyStopping()]
pch_log = pch_model.fit(X_train, Y_train_pch, batch_size, epochs, callbacks, val_data=(X_val, Y_val_pch))
pch_log.plot();

In [ ]:
torch.manual_seed(42)

dnn_model = tt.practical.MLPVanilla(in_features  = num_inputs,
                                    num_nodes    = hidden_nodes,
                                    out_features = num_outputs)
optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
pmf_model  = PMF(dnn_model, optimizer, duration_index=pmf_labtrans.cuts)
callbacks  = [tt.cb.EarlyStopping()]
pmf_log = pmf_model.fit(X_train, Y_train_pmf, batch_size, epochs, callbacks, val_data=(X_val, Y_val_pmf))
pmf_log.plot();

In [ ]:
torch.manual_seed(42)

dnn_model = tt.practical.MLPVanilla(in_features  = num_inputs,
                                    num_nodes    = hidden_nodes,
                                    out_features = num_outputs)
optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
mtlr_model = MTLR(dnn_model, optimizer, duration_index=mtlr_labtrans.cuts)
callbacks  = [tt.cb.EarlyStopping()]
mtlr_log = mtlr_model.fit(X_train, Y_train_mtlr, batch_size, epochs, callbacks, val_data=(X_val, Y_val_mtlr))
mtlr_log.plot();

In [ ]:
torch.manual_seed(42)

dnn_model = tt.practical.MLPVanilla(in_features  = num_inputs,
                                    num_nodes    = hidden_nodes,
                                    out_features = num_outputs)
optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
bces_model = BCESurv(dnn_model, optimizer, duration_index=bces_labtrans.cuts)
callbacks  = [tt.cb.EarlyStopping()]
bces_log = bces_model.fit(X_train, Y_train_bces, batch_size, epochs, callbacks, val_data=(X_val, Y_val_bces))
bces_log.plot();

In [ ]:
torch.manual_seed(42)

dnn_model = tt.practical.MLPVanilla(in_features  = num_inputs,
                                    num_nodes    = hidden_nodes,
                                    out_features = num_outputs)
optimizer = optim.Adam(dnn_model.parameters(), lr=0.01)
dhs_model  = DeepHitSingle(dnn_model, optimizer, duration_index=dhs_labtrans.cuts)
callbacks  = [tt.cb.EarlyStopping()]
dhs_log = dhs_model.fit(X_train, Y_train_bces, batch_size, epochs, callbacks, val_data=(X_val, Y_val_dhs))
dhs_log.plot();

## **6. Prediction: Survival Estimates**

- `model.predict_surv` returns an array of survival estimates

- `model.predict_surv_df` returns the survival estimates as a dataframe.

In [ ]:
lh_pred_surv_train = lh_model.predict_surv_df(X_train)
lh_pred_surv_val = lh_model.predict_surv_df(X_val)
lh_pred_surv_test = lh_model.predict_surv_df(X_test)
lh_pred_surv_test

- Survival estimates for the first 5 individuals.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
lh_pred_surv_test.iloc[: , :5].plot()
plt.ylabel('Survival Probability')
plt.xlabel('Time');

- By default, linear interpolation is performed as show in the figure above.

- However, the survival estimates are only defined at the 10 time points in the discretization grid.

- The survival estimates for each individual is a step function.

In [ ]:
lh_pred_surv_test.iloc[: , :5].plot(drawstyle='steps-post')
plt.ylabel('Survival Probability')
plt.xlabel('Time');

## **7. Model Evaluation**

- The `EvalSurv` class contains some useful evaluation criteria for time-to-event prediction.

- Use Kaplan-Meier by default for estimating the censoring distribution.

In [ ]:
from pycox.evaluation import EvalSurv

### **i. Concordance Index (C-index)**

- `.concordance_td()`

    - Time-dependent concordance index (higher = better).

    - Requires setting `censor_surv='km'` for obtaining Integrated Brier Score (IBS).

- It measures how well the model orders survival times.

In [ ]:
train_eval = EvalSurv(lh_pred_surv_train, *Y_train, censor_surv='km')
train_eval.concordance_td()

In [ ]:
val_eval = EvalSurv(lh_pred_surv_val, *Y_val, censor_surv='km')
val_eval.concordance_td()

In [ ]:
test_eval = EvalSurv(lh_pred_surv_test, *Y_test, censor_surv='km')
test_eval.concordance_td()

### **ii. Brier Scores (BSs)**

- Measure calibration and discrimination by comparing predicted survival probability with observed outcomes.

- Lower = better.

- Requires choosing a time grid (set of times to evaluate).
    
    `ev.brier_score(time_grid)`




In [ ]:
time_grid = np.linspace(np.array(Y_df["SurvTime"]).min(), np.array(Y_df["SurvTime"]).max(), 100)
# time_grid

In [ ]:
import scipy.integrate as integrate

In [ ]:
train_eval.brier_score(time_grid)

- Plot the Brier scores for a given set of times.

In [ ]:
test_eval.brier_score(time_grid).plot()
plt.ylabel('Brier Scores')
plt.xlabel('Time');

### **iii. Integrated Brier Score (IBS)**

- Measure the overall calibration and discrimination by comparing predicted survival probability with observed outcomes.

- Lower = better.

- Requires choosing a time grid (set of times to evaluate).
    
    `ev.integrated_brier_score(time_grid)`

In [ ]:
if not hasattr(integrate, "simps"):
    integrate.simps = integrate.simpson

test_eval.integrated_brier_score(time_grid)

### **iv. Negative Binomial Log-Likelihood (NLL) (a.k.a. Log Scores)**

- It measures how well predicted survival curves match actual outcomes in terms of likelihood.

- Lower = better.

In [ ]:
test_eval.nbll(time_grid)

In [ ]:
test_eval.nbll(time_grid).plot()
plt.ylabel('NBLL')
plt.xlabel('Time');

### **v. Integrated NBLL**

In [ ]:
test_eval.integrated_nbll(time_grid)

**When to use which?**

- C-index: Ranking ability (discrimination).

- NBLL / Log score: Good for probabilistic performance.

- IBS: Calibration & overall performance (more reliable than C-index alone).

## **II. Continuous Time Models**

- To be tried by yourself.

**CoxPH (DeepSurv)**

- This is the deep version of the classical Cox Proportional Hazards model.

`y = (durations, events)`

`model.fit(x, y, batch_size, epochs, callbacks)`

`cindex = model.score(x, y)`

**CoxTime**

- This model allows time-varying effects — the network learns a time-dependent representation of risk.

- When hazards are non-proportional.

`labtrans = LabTransCoxTime()`

`y = labtrans.fit_transform(durations, events)`

`model.fit(x, y, batch_size, epochs, callbacks)`

**CoxCC (Case-Control Cox)**

- This model approximates the Cox likelihood using case–control sampling.

- It’s similar to CoxTime but computationally lighter for large datasets.


`labtrans = LabTransCoxTime()  # same label transform as CoxTime`

`y = labtrans.fit_transform(durations, events)`

`model.fit(x, y, batch_size, epochs, callbacks)`